In [6]:
# ============================================================
# CELA 1 – KONFIGURACJA I DATY
# ============================================================
import win32com.client
import time
from datetime import datetime, timedelta
import os

# --- KONFIGURACJA ---
MATERIAL = "00204321-01-0000"
PLANT = "CZ01"
SAVE_PATH = r"C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04"

# Obliczanie dat
today = datetime.today()
monday = today - timedelta(days=today.weekday())       # Poniedziałek bieżącego tygodnia
end_of_year = datetime(today.year, 12, 31)             # Ostatni dzień roku

date_from = monday.strftime("%d.%m.%Y")
date_to   = end_of_year.strftime("%d.%m.%Y")
date_file = today.strftime("%d.%m.%Y")

print(f"Data początkowa (poniedziałek): {date_from}")
print(f"Data końcowa (koniec roku):     {date_to}")
print(f"Data do nazwy pliku:            {date_file}")
print(f"Ścieżka zapisu:                 {SAVE_PATH}")

os.makedirs(SAVE_PATH, exist_ok=True)


Data początkowa (poniedziałek): 30.03.2026
Data końcowa (koniec roku):     31.12.2026
Data do nazwy pliku:            03.04.2026
Ścieżka zapisu:                 C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04


In [7]:
# ============================================================
# CELA 2 – POŁĄCZENIE Z SAP
# ============================================================
SapGuiAuto = win32com.client.GetObject("SAPGUI")
application = SapGuiAuto.GetScriptingEngine
connection  = application.Children(0)   # Pierwsze połączenie
session     = connection.Children(0)    # Pierwsza sesja

print(f"Połączono z SAP!")
print(f"System: {connection.Description}")
print(f"Sesja:  {session.Info.SystemName}")
print(f"User:   {session.Info.User}")
print(f"Transakcja: {session.Info.Transaction}")


Połączono z SAP!
System: ECP Production (001)
Sesja:  ECP
User:   JNIEDZWI1
Transakcja: SESSION_MANAGER


In [15]:
# ============================================================
# CELA 3 – HELPER: ODKRYCIE FIELD IDs (uruchom raz)
# Uruchom będąc na ekranie ZDTS_MD04 – pokaże techniczne nazwy pól
# ============================================================
def discover_fields(container, max_depth=6, depth=0):
    results = []
    if depth > max_depth:
        return results
    try:
        children = container.Children
        for i in range(children.Count):
            child = children(i)
            child_text = getattr(child, 'Text', '')
            if len(str(child_text)) > 80:
                child_text = str(child_text)[:80] + "..."
            results.append({
                'id':   child.Id,
                'type': child.Type,
                'text': child_text
            })
            results.extend(discover_fields(child, max_depth, depth + 1))
    except Exception:
        pass
    return results

session.StartTransaction("ZDTS_MD04")
time.sleep(1)

fields = discover_fields(session.FindById("wnd[0]/usr"))

print(f"Znaleziono {len(fields)} elementów:\n")
print(f"{'TYP':<30} {'ID':<60} {'TEXT'}")
print("-" * 150)
for f in fields:
    if f['type'] in ('GuiTextField', 'GuiCTextField', 'GuiRadioButton',
                     'GuiCheckBox', 'GuiButton', 'GuiComboBox'):
        print(f"{f['type']:<30} {f['id']:<60} {f['text']}")


Znaleziono 50 elementów:

TYP                            ID                                                           TEXT
------------------------------------------------------------------------------------------------------------------------------------------------------
GuiTextField                   /app/con[0]/ses[0]/wnd[0]/usr/txt%_S_BESKZ_%_APP_%-TEXT      Procurement Type
GuiCTextField                  /app/con[0]/ses[0]/wnd[0]/usr/ctxtS_BESKZ-LOW                
GuiButton                      /app/con[0]/ses[0]/wnd[0]/usr/btn%_S_BESKZ_%_APP_%-VALU_PUSH 
GuiTextField                   /app/con[0]/ses[0]/wnd[0]/usr/txt%_S_MDV01_%_APP_%-TEXT      Work Center
GuiCTextField                  /app/con[0]/ses[0]/wnd[0]/usr/ctxtS_MDV01-LOW                
GuiButton                      /app/con[0]/ses[0]/wnd[0]/usr/btn%_S_MDV01_%_APP_%-VALU_PUSH 
GuiTextField                   /app/con[0]/ses[0]/wnd[0]/usr/txt%_S_DISPO_%_APP_%-TEXT      MRP Controller
GuiCTextField                  /ap

In [16]:
# ============================================================
# CELA 4 – MAPOWANIE PÓŁ
# Po uruchomieniu Celi 3 podmień ID na te z wyników!
# ============================================================
FIELDS = {
    # --- Characteristics ---
    "material_from":   "/app/con[0]/ses[0]/wnd[0]/usr/ctxtS_MATNR-LOW",
    "plant":           "/app/con[0]/ses[0]/wnd[0]/usr/ctxtS_WERKS-LOW",
    # --- Kind of Report ---
    "only_weekly":     "/app/con[0]/ses[0]/wnd[0]/usr/radP_WEEK",
    "14_days_daily":   "/app/con[0]/ses[0]/wnd[0]/usr/radP_FORT",
    # --- Period to Analyse ---
    "date_from":       "/app/con[0]/ses[0]/wnd[0]/usr/ctxtS_DATE-LOW",
    "date_to":         "/app/con[0]/ses[0]/wnd[0]/usr/ctxtS_DATE-HIGH",
    # --- Additional Information ---
    "with_cust_req":   "/app/con[0]/ses[0]/wnd[0]/usr/chkP_WCUST",   # With Customer Requirement
    "with_stock_info": "/app/con[0]/ses[0]/wnd[0]/usr/chkP_WSTOCK",  # With Stock Information
    "only_cust_req":   "/app/con[0]/ses[0]/wnd[0]/usr/chkP_OCUST",   # Only Customer Requirement
}

print("Mapowanie pól załadowane.")
print("WAŻNE: Jeśli Cela 3 pokazała inne ID – podmień wartości powyżej!")


Mapowanie pól załadowane.
WAŻNE: Jeśli Cela 3 pokazała inne ID – podmień wartości powyżej!


In [17]:
# ============================================================
# CELA 5 – FUNKCJE POMOCNICZE
# ============================================================
BASE = "/app/con[0]/ses[0]"

import threading
import win32gui
import win32con

def fill_zdts_md04(session, material, plant, date_from, date_to, report_type="weekly"):
    session.StartTransaction("ZDTS_MD04")
    time.sleep(2)
    session.FindById(FIELDS["material_from"]).Text = material
    session.FindById(FIELDS["plant"]).Text = plant
    if report_type == "weekly":
        session.FindById(FIELDS["only_weekly"]).Select()
    elif report_type == "14days":
        session.FindById(FIELDS["14_days_daily"]).Select()
    session.FindById(FIELDS["date_from"]).Text = date_from
    session.FindById(FIELDS["date_to"]).Text = date_to
    chk_cust = session.FindById(FIELDS["with_cust_req"])
    if not chk_cust.Selected:
        chk_cust.Selected = True
    chk_stock = session.FindById(FIELDS["with_stock_info"])
    if not chk_stock.Selected:
        chk_stock.Selected = True
    chk_only = session.FindById(FIELDS["only_cust_req"])
    if chk_only.Selected:
        chk_only.Selected = False
    print(f"[OK] Formularz [{report_type}]: {material} | {plant} | {date_from} -> {date_to}")
    print(f"     With Customer Req: ON | With Stock Info: ON | Only Customer Req: OFF")


def _diagnose_wnd(session, wnd_idx=1):
    def _show(container, depth=0):
        if depth > 4:
            return
        try:
            for i in range(container.Children.Count):
                child = container.Children(i)
                text = str(getattr(child, 'Text', ''))[:60]
                print("  " * depth + f"{child.Type:<28} {child.Id}  '{text}'")
                _show(child, depth + 1)
        except Exception:
            pass
    try:
        wnd = session.FindById(f"{BASE}/wnd[{wnd_idx}]")
        print(f"\n--- wnd[{wnd_idx}]: '{wnd.Text}' ---")
        _show(wnd)
    except Exception:
        print(f"Brak wnd[{wnd_idx}]")


def _auto_accept_sap_security(duration=30):
    """
    Watek w tle: szuka okna 'Bezpieczenstwo SAP GUI' i klika 'Zezwalanie'.
    Dziala przez 'duration' sekund. Obsluguje wiele dialogow pod rzad.
    """
    end_time = time.time() + duration
    clicked = 0
    while time.time() < end_time:
        try:
            hwnd = win32gui.FindWindow(None, "Bezpieczeństwo SAP GUI")
            if hwnd and win32gui.IsWindowVisible(hwnd):
                # Szukaj przycisku "Zezwalanie" wsrod dzieci okna
                def enum_children(child_hwnd, buttons):
                    try:
                        cls = win32gui.GetClassName(child_hwnd)
                        txt = win32gui.GetWindowText(child_hwnd)
                        if cls == "Button" and "Zezwalanie" in txt:
                            buttons.append(child_hwnd)
                    except Exception:
                        pass
                    return True
                buttons = []
                win32gui.EnumChildWindows(hwnd, enum_children, buttons)
                if buttons:
                    win32gui.PostMessage(buttons[0], win32con.BM_CLICK, 0, 0)
                    clicked += 1
                    print(f"[OK] Auto-klik 'Zezwalanie' (dialog #{clicked})")
                    time.sleep(2)  # Odczekaj zanim szukamy nastepnego
        except Exception:
            pass
        time.sleep(0.5)


def run_report_and_save(session, save_path, filename):
    """Uruchamia raport (F8), czeka na ALV grid, eksportuje do xlsx."""

    session.FindById(f"{BASE}/wnd[0]/tbar[1]/btn[8]").Press()
    print("Wykonuje raport (F8)...")
    time.sleep(5)

    GRID_ID = f"{BASE}/wnd[0]/usr/cntlCONT/shellcont/shell"
    print("Czekam na zaladowanie wynikow...")
    for attempt in range(30):
        try:
            session.FindById(GRID_ID)
            print(f"[OK] Raport zaladowany (proba {attempt + 1})")
            break
        except Exception:
            time.sleep(2)
    else:
        print("UWAGA: Grid nie zaladowal sie w 60s")

    time.sleep(2)
    shell = session.FindById(GRID_ID)

    # --- Uruchom watek w tle ktory bedzie klikal "Zezwalanie" przez 30s ---
    security_thread = threading.Thread(
        target=_auto_accept_sap_security,
        args=(30,),
        daemon=True
    )
    security_thread.start()
    print("[OK] Watek bezpieczenstwa uruchomiony (30s)")

    # --- Eksport dropdown &MB_EXPORT -> &XXL ---
    exported = False
    for submenu_id in ["&EXPORT_SPREADSHEET", "&XXL", "&MB_EXPORT_SPREADSHEET"]:
        try:
            shell.PressToolbarContextButton("&MB_EXPORT")
            time.sleep(1)
            shell.SelectContextMenuItem(submenu_id)
            print(f"[OK] Export -> '{submenu_id}'")
            exported = True
            time.sleep(2)
            break
        except Exception as e:
            print(f"[INFO] {submenu_id}: {e}")

    if not exported:
        for menu_id in ["&EXPORT_SPREADSHEET", "&XXL", "&EXPORT_LOCAL_FILE"]:
            try:
                shell.SelectContextMenuItem(menu_id)
                print(f"[OK] Context menu: '{menu_id}'")
                exported = True
                time.sleep(2)
                break
            except Exception as e:
                print(f"[INFO] {menu_id}: {e}")

    if not exported:
        print("[ERR] Wszystkie metody eksportu zawiodly")
        return False

    # --- Dialog formatu (jesli sie pojawi) ---
    try:
        wnd1 = session.FindById(f"{BASE}/wnd[1]")
        print(f"[OK] Dialog: '{wnd1.Text}'")
        for idx in range(10):
            try:
                radio = session.FindById(
                    f"{BASE}/wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150"
                    f"/sub:SAPLSPO5:0150/radSPOPLI-SELFLAG[{idx},0]"
                )
                label = str(getattr(radio, 'Text', ''))
                if 'spreadsheet' in label.lower():
                    radio.Select()
                    print(f"  -> Format: '{label}'")
                    break
            except Exception:
                break
        session.FindById(f"{BASE}/wnd[1]/tbar[0]/btn[0]").Press()
        time.sleep(2)
        print("[OK] Format potwierdzony")
    except Exception:
        print("Brak dialogu formatu")

    # --- Dialog zapisu (sciezka + nazwa pliku) ---
    time.sleep(2)
    try:
        wnd1 = session.FindById(f"{BASE}/wnd[1]")
        print(f"[OK] Dialog zapisu: '{wnd1.Text}'")
        session.FindById(f"{BASE}/wnd[1]/usr/ctxtDY_PATH").Text = save_path
        session.FindById(f"{BASE}/wnd[1]/usr/ctxtDY_FILENAME").Text = filename
        print(f"  Sciezka: {save_path}")
        print(f"  Plik:    {filename}")
        session.FindById(f"{BASE}/wnd[1]/tbar[0]/btn[11]").Press()
        # Watek w tle automatycznie kliknie "Zezwalanie" w dialogach bezpieczenstwa
        time.sleep(5)
        print(f"[OK] Zapisano: {os.path.join(save_path, filename)}")
        return True
    except Exception as e:
        print(f"[ERR] Dialog zapisu: {e}")
        _diagnose_wnd(session, 1)
        return False


print("Funkcje zaladowane.")

Funkcje zaladowane.


In [18]:
# ============================================================
# CELA 6 – RAPORT 1: WEEKLY
# ============================================================
print("=" * 60)
print("RAPORT 1: Only Weekly")
print("=" * 60)

fill_zdts_md04(
    session=session, material=MATERIAL, plant=PLANT,
    date_from=date_from, date_to=date_to, report_type="weekly"
)

filename_weekly = f"{date_file} week.xlsx"
run_report_and_save(session, SAVE_PATH, filename_weekly)
print(f"\nZapisano jako: {filename_weekly}")


RAPORT 1: Only Weekly
[OK] Formularz [weekly]: 00204321-01-0000 | CZ01 | 30.03.2026 -> 31.12.2026
     With Customer Req: ON | With Stock Info: ON | Only Customer Req: OFF
Wykonuje raport (F8)...
Czekam na zaladowanie wynikow...
[OK] Raport zaladowany (proba 1)
[OK] Watek bezpieczenstwa uruchomiony (30s)
[INFO] &EXPORT_SPREADSHEET: (-2147352567, 'Wystąpił wyjątek.', (613, 'SAP Frontend Server', 'The method got an invalid argument.', None, 0, 0), None)
[OK] Export -> '&XXL'
[OK] Dialog: 'Material Planning Report'
[OK] Format potwierdzony
[OK] Dialog zapisu: 'Material Planning Report'
  Sciezka: C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04
  Plik:    03.04.2026 week.xlsx
[OK] Zapisano: C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04\03.04.2026 week.xlsx

Zapisano jako: 03.04.2026 week.xlsx


In [19]:
# ============================================================
# CELA 7 – RAPORT 2: 14 DAYS DAILY / REST WEEKLY
# ============================================================
print("=" * 60)
print("RAPORT 2: 14 Days Daily / Rest Weekly")
print("=" * 60)

fill_zdts_md04(
    session=session, material=MATERIAL, plant=PLANT,
    date_from=date_from, date_to=date_to, report_type="14days"
)

filename_14days = f"{date_file} 14 days.xlsx"
run_report_and_save(session, SAVE_PATH, filename_14days)
print(f"\nZapisano jako: {filename_14days}")


RAPORT 2: 14 Days Daily / Rest Weekly
[OK] Formularz [14days]: 00204321-01-0000 | CZ01 | 30.03.2026 -> 31.12.2026
     With Customer Req: ON | With Stock Info: ON | Only Customer Req: OFF
Wykonuje raport (F8)...
Czekam na zaladowanie wynikow...
[OK] Raport zaladowany (proba 1)
[OK] Watek bezpieczenstwa uruchomiony (30s)
[INFO] &EXPORT_SPREADSHEET: (-2147352567, 'Wystąpił wyjątek.', (613, 'SAP Frontend Server', 'The method got an invalid argument.', None, 0, 0), None)
[OK] Export -> '&XXL'
[OK] Dialog: 'Material Planning Report'
[OK] Format potwierdzony
[OK] Dialog zapisu: 'Material Planning Report'
  Sciezka: C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04
  Plik:    03.04.2026 14 days.xlsx
[OK] Zapisano: C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04\03.04.2026 14 days.xlsx

Zapisano jako: 03.04.2026 14 days.xlsx


In [13]:
# ============================================================
# CELA 8 – POWRÓT DO ZDTS_MD04 (gotowy na kolejne kroki)
# ============================================================
session.StartTransaction("ZDTS_MD04")
time.sleep(2)

print("Gotowe! SAP jest na ekranie ZDTS_MD04.")
print(f"\nZapisane pliki:")
print(f"  1. {os.path.join(SAVE_PATH, filename_weekly)}")
print(f"  2. {os.path.join(SAVE_PATH, filename_14days)}")


Gotowe! SAP jest na ekranie ZDTS_MD04.

Zapisane pliki:
  1. C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04\03.04.2026 week.xlsx
  2. C:\Users\jakub.niedzwiadek2\OneDrive - OPmobility\Pulpit\Anita Sankowska\ZDTS_MD04\03.04.2026 14 days.xlsx
